# AdaLoop-VLM test notebook (stage S0: model and pruning)
Each section calls `adaloop.api` and asserts. Pass = every `PASS` line prints.

In [ ]:
CONFIG = "configs/mvp.yaml"

In [ ]:
import logging
import sys
from pathlib import Path

import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")

from adaloop import api


def expect_error(fn, error=ValueError):
    try:
        fn()
    except error as exc:
        print("raised as expected:", type(exc).__name__, exc)
        return
    raise AssertionError(f"expected {error.__name__}")

## Load
Pass: no missing or unexpected keys, nothing on CPU or disk.

In [ ]:
print("[STAGE] load")
cfg = api.load_config(ROOT / CONFIG)
vlm = api.load_model(cfg)
print(vlm.model.hf_device_map)
assert not vlm.loading_info["missing_keys"] and not vlm.loading_info["unexpected_keys"]
assert all(str(d) not in ("cpu", "disk", "meta") for d in vlm.model.hf_device_map.values())
print("PASS S0: model loads fully on GPU")

## Inputs

In [ ]:
print("[STAGE] inputs")
URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
question = "What animal is on the candy?"
image = api.load_image(URL, cfg["res_px"])
inputs = api.prepare_inputs(vlm, question, image)
text_only = api.prepare_inputs(vlm, "What is 12 multiplied by 5?")
display(image)
print({k: tuple(v.shape) for k, v in inputs.items()})
expect_error(lambda: api.load_image("missing.png", cfg["res_px"]), FileNotFoundError)
print("PASS S0: inputs built")

## Loops
Pass: layer calls = 6 + 12R + 6 and logits differ between R=1 and R=3.

In [ ]:
print("[STAGE] loops")
for R in range(1, 6):
    api.set_loops(vlm, R)
    calls = api.count_layer_calls(vlm, inputs)
    print(R, sum(calls), calls)
    assert calls == api.expected_layer_calls(vlm) and sum(calls) == 6 + 12 * R + 6
api.set_loops(vlm, 1)
logits_r1 = api.last_logits(vlm, inputs)
api.set_loops(vlm, 3)
logits_r3 = api.last_logits(vlm, inputs)
assert not torch.equal(logits_r1, logits_r3)
for bad in (0, 6):
    expect_error(lambda: api.set_loops(vlm, bad))
print("PASS S0: loop count is controlled at runtime")

## Inference

In [ ]:
print("[STAGE] inference")
api.set_loops(vlm, 3)
for batch in (text_only, inputs):
    result = api.generate(vlm, batch, cfg["max_new_tokens"])
    print(result.text, "\n---")
    assert result.text.strip()
print("PASS S0: one answer generated")

## Pruning
Pass: k=100% reproduces the unpruned logits exactly; kept counts match k.

In [ ]:
print("[STAGE] pruning")
api.set_loops(vlm, 3)
visual, vision_out = api.extract_visual_tokens(vlm, inputs)
n = len(visual)
full = api.prune_inputs(vlm, inputs, vision_out, api.random_selector(visual, 1.0))
original, rebuilt = api.last_logits(vlm, inputs), api.last_logits(vlm, full)
assert torch.equal(original, rebuilt), f"max diff {(original - rebuilt).abs().max().item()}"
print("PASS S0: k=100% is bit-exact")

question_emb = api.question_embeddings(vlm, question)
for k in (0.5, 0.25):
    selected = {
        "random": api.random_selector(visual, k),
        "uniform": api.uniform_selector(inputs["image_grid_thw"], k),
        "text_mean": api.text_mean_selector(visual, question_emb, k)[0],
    }
    for name, idx in selected.items():
        assert len(idx) == max(1, int(n * k)) and torch.equal(idx, idx.unique())
        pruned = api.prune_inputs(vlm, inputs, vision_out, idx)
        assert pruned["input_ids"].shape[-1] == inputs["input_ids"].shape[-1] - n + len(idx)
        print(k, name, len(idx), api.generate(vlm, pruned, cfg["max_new_tokens"]).text[:200])
print("PASS S0: selectors and pruning")

expect_error(lambda: api.random_selector(visual, 0))
expect_error(lambda: api.random_selector(visual, 1.5))
expect_error(lambda: api.prune_inputs(vlm, inputs, vision_out, torch.tensor([1, 0])))
expect_error(lambda: api.prune_inputs(vlm, inputs, vision_out, torch.tensor([0, 0])))
expect_error(lambda: api.extract_visual_tokens(vlm, text_only))
print("PASS S0: bad inputs rejected")